In [ ]:

import urllib.parse, urllib.request, time
import xml.etree.ElementTree as ET
import requests
from tqdm import tqdm

NS = {"a": "http://www.w3.org/2005/Atom"}
CAT = (
    "(cat:math.* OR cat:stat.*"
    " OR cat:physics.*"
    " OR cat:cond-mat.*"
    " OR cat:quant-ph"
    " OR cat:hep-th OR cat:hep-ph OR cat:hep-ex OR cat:hep-lat"
    " OR cat:gr-qc"
    " OR cat:nucl-th OR cat:nucl-ex"
    " OR cat:astro-ph.*"
    " OR cat:nlin.*"
    " OR cat:eess.*"
    " OR cat:q-bio.*"
    " OR cat:q-fin.*"
    ")"
)
API = "https://export.arxiv.org/api/query"

title_phrases = [
    "open problems", "open problem",
    "open questions", "open question",
    "unsolved problems", "unsolved problem",
    "open conjectures",
    "problems and conjectures",
    "list of problems","list of open problems",
    "problem list",
    "problem collection", "problem book",
    "collection of problems",
    "problem session", "problem sessions",
    "problem workshop",
    "research problems",
    "ten problems", "few problems",
]

def api_page(search_query, start, page_size=200, max_retries=5):
    qs = urllib.parse.urlencode({
        "search_query": search_query,
        "start": start,
        "max_results": page_size,
        "sortBy": "submittedDate",
        "sortOrder": "descending",
    })
    req = urllib.request.Request(
        f"{API}?{qs}",
        headers={"User-Agent": "open-problem-harvester/1.0 (kevinjk02@snu.ac.kr)"}
    )
    for attempt in range(max_retries):
        try:
            with urllib.request.urlopen(req, timeout=60) as r:
                return r.read().decode()
        except urllib.error.HTTPError as e:
            if e.code == 429:
                wait = 10 * (2 ** attempt)  # 10s, 20s, 40s, 80s, 160s
                print(f"  429 rate-limited, waiting {wait}s (attempt {attempt+1}/{max_retries})...")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError(f"Failed after {max_retries} retries (429 rate limit)")

def fetch_phrase(phrase, page_size=200, max_pages=5, delay=5.0):
    q = f'ti:"{phrase}" AND {CAT}'
    out, start = [], 0
    for _ in range(max_pages):
        xml = api_page(q, start, page_size)
        root = ET.fromstring(xml)
        entries = root.findall("a:entry", NS)
        for e in entries:
            tid   = e.findtext("a:id", "", NS).strip()
            title = " ".join(e.findtext("a:title", "", NS).split())
            abs_url = tid.replace("http://arxiv.org/abs/", "https://arxiv.org/abs/")
            abs_url = abs_url.rsplit("v", 1)[0] if "v" in abs_url.split("/")[-1] else abs_url
            out.append((abs_url, title))
        time.sleep(delay)
        if len(entries) < page_size:
            break
        start += page_size
    return out

rows = []
for phrase in tqdm(title_phrases):
    rows.extend(fetch_phrase(phrase))

# dedupe by link, keep first title seen
seen, links, titles = set(), [], []
for url, t in rows:
    if url in seen:
        continue
    seen.add(url)
    links.append(url)
    titles.append(t)

print(f"raw hits: {len(rows)}  unique links: {len(links)}")

100%|██████████| 20/20 [02:15<00:00,  6.76s/it]

raw hits: 1206  unique links: 631


In [2]:
links = list(set(links))

In [40]:
from tqdm import tqdm
titles = []
for link in tqdm(links):
    try:
        html = requests.get(link)
        bs4_obj = BeautifulSoup(html.text,'html.parser')
        title = bs4_obj.find('div',{'id':'content-inner'}).h1.text
        titles.append(title)
    except:
        continue

100%|██████████| 427/427 [01:43<00:00,  4.13it/s]


In [3]:
import pandas as pd
df = pd.DataFrame({'title':titles,'links':links})

In [4]:
df.sample(5)

,title,links
503,Open questions in quarkonium and electromagnet...,https://arxiv.org/abs/0911.4460
528,"Unsolved problems on joinings, multiple mixing...",https://arxiv.org/abs/2312.03907
209,Open problems on k-orbit polytopes,https://arxiv.org/abs/hep-th/0104131
453,Infinite dimensional moment problem: open ques...,https://arxiv.org/abs/1507.05308
589,A list of problems in Plane Geometry with simp...,https://arxiv.org/abs/1605.07477


In [ ]:
df.to_csv('arxiv-op-papers-0501.csv', index=False)

In [16]:
from litellm import completion
import os
# Set OPENAI_API_KEY in your environment before running this cell

from openai import OpenAI
client = OpenAI()

system_prompt = """
You will be given a link of an arxiv paper.
Im looking for papers that list open problems that are unsolved. 
Your job is to open the link to validate whether the provided paper meets my expectations.
Return either True or False only.
"""

output = []
for _,row in tqdm(df.iterrows(),total=len(df)):
    response = client.responses.create(
        model="gpt-5-mini",
        tools=[{"type": "web_search"}],
        input=system_prompt + f"use this link {row.links}"
    )
    
    output.append(response.output_text)


100%|██████████| 631/631 [2:26:53<00:00, 13.97s/it]  


In [ ]:
import pandas as pd

df_val = pd.read_csv('arxiv-op-papers-0501.csv')
df_val['is_open_problem_list'] = output

df_val = df_val[df_val['is_open_problem_list'].astype(str).str.contains('True')]
df_val = df_val.drop(columns=['is_open_problem_list'])

df_val.to_csv('arxiv-op-papers-validated.csv', index=False)

In [52]:
print(df.generation.values[0].split('assistantfinal')[1])

{
  "question_style": "proof",
  "difficulty": "research",
  "is_self_contained": true,
  "self_contained_explanation": "The problem statement defines all geometric objects (planar point set, triangulation, edge length, total weight) and the decision version of the optimization problem, so no external concepts are required beyond standard notions of polynomial time and NP‑completeness.",
  "subject_area": "Computational Geometry",
  "subfields": [
    "Geometric Algorithms",
    "Computational Complexity",
    "Combinatorial Optimization"
  ],
  "core_concepts": [
    "Triangulation of a planar point set",
    "Edge length and total weight of a triangulation",
    "Optimization (minimum total weight)",
    "Decision version of an optimization problem",
    "Polynomial‑time algorithms",
    "NP‑completeness and NP‑hardness"
  ],
  "required_theorems_or_results": [
    "Fundamental NP‑completeness theory (Cook‑Levin theorem, polynomial‑time reductions)",
    "Existence of triangulations 

In [34]:
# === zbMATH title-search scraper ===
# Scrapes zbmath.org result pages for papers whose title matches open-problem phrases.
# zbMATH entries link to the source paper (often arxiv/journal), which we capture.
import requests, time, re
from bs4 import BeautifulSoup
from urllib.parse import quote_plus, urljoin
from tqdm import tqdm
import pandas as pd

ZB = "https://zbmath.org"
UA = {"User-Agent": "Mozilla/5.0 (open-problem-harvester)"}

zb_phrases = [
    "open problems", "open questions", "unsolved problems",
    "problem session", "problem collection", "problem list",
    "research problems", "problems and conjectures",
    "list of problems", "problem book",
]

def zb_fetch(phrase, max_pages=5, delay=2.5):
    rows, seen_entry = [], set()
    q = quote_plus(f'ti:"{phrase}"')
    for page in range(1, max_pages + 1):
        url = f"{ZB}/?q={q}&p={page}"
        try:
            r = requests.get(url, timeout=60, headers=UA)
        except Exception as e:
            print(f"[zb] {phrase} p{page}: {e}")
            break
        if r.status_code != 200:
            print(f"[zb] {phrase} p{page}: HTTP {r.status_code}")
            break
        soup = BeautifulSoup(r.text, "html.parser")
        # Each result is a block; pull the Zbl entry link + visible title + any external link (arxiv/DOI)
        items = soup.select("div.list > div, div.results-list div.item")
        if not items:
            # fallback: walk all anchors to entry pages
            items = [soup]
        hit = 0
        for it in items:
            a_entry = it.find("a", href=re.compile(r"q=an(%3A|:)"))
            if not a_entry:
                continue
            entry = urljoin(ZB, a_entry["href"])
            if entry in seen_entry:
                continue
            seen_entry.add(entry)
            title = " ".join(a_entry.get_text(" ").split())
            # external links inside the same item (arxiv / doi / publisher)
            ext = []
            for a in it.find_all("a", href=True):
                h = a["href"]
                if "arxiv.org" in h or "doi.org" in h:
                    ext.append(h)
            rows.append({"entry": entry, "title": title, "external": "|".join(ext), "phrase": phrase})
            hit += 1
        if hit == 0:
            break
        time.sleep(delay)
    return rows

zb_rows = []
for p in tqdm(zb_phrases):
    zb_rows.extend(zb_fetch(p))

zb_df = pd.DataFrame(zb_rows).drop_duplicates("entry") if zb_rows else pd.DataFrame(columns=["entry","title","external","phrase"])
print(f"zbmath unique entries: {len(zb_df)}")
zb_df.head(10)


100%|██████████| 10/10 [00:27<00:00,  2.79s/it]

zbmath unique entries: 0


,entry,title,external,phrase


In [ ]:
# === Google Search harvester (SerpAPI REST) ===

import os, time, requests
from tqdm import tqdm
import pandas as pd

SERPAPI_KEY = os.environ.get("SERPAPI_KEY", "")

google_queries = [
    "open problems pdf",
    "open questions pdf",
    "unsolved problems pdf",
    "open conjectures pdf",
    "list of open problems pdf",
    "research problems pdf",
    "problems and conjectures pdf",
    "problem collection pdf",
]

NUM_PAGES = 10  

g_rows = []
for q in tqdm(google_queries):
    for page in range(NUM_PAGES):
        try:
            r = requests.get("https://serpapi.com/search", params={
                "engine": "google",
                "q": q,
                "start": page * 10,
                "api_key": SERPAPI_KEY,
            }, timeout=30)
            r.raise_for_status()
            data = r.json()
            if "error" in data:
                print(f"[serpapi] {q} p{page}: {data['error']}")
                break
            items = data.get("organic_results", [])
            for item in items:
                url = item.get("link", "")
                title = item.get("title", "")
                if url:
                    g_rows.append({"title": title, "url": url})
            if len(items) < 10: 
                break
        except Exception as e:
            print(f"[serpapi] {q} p{page}: {e}")
            break
        time.sleep(1)

g_df = pd.DataFrame(g_rows).drop_duplicates("url") if g_rows else pd.DataFrame(columns=["title", "url"])
print(f"google unique urls: {len(g_df)}")
g_df.head(10)

100%|██████████| 11/11 [00:59<00:00,  5.40s/it]

google unique urls: 458


,title,url
0,"100 OPEN PROBLEMS Contents 1. Sum-free sets, p...",https://people.maths.ox.ac.uk/greenbj/papers/o...
1,A List of Open Problems Compiled by,https://aimath.org/WWN/polyaschurlax/polyaschu...
2,Open Problems and Projects Stephen Wolfram,https://www.wolframscience.com/openproblems/NK...
3,Some of my favorite open problems,https://www.stat.berkeley.edu/~aldous/Talks/OP...
4,open problems in dynamics and related fields,https://www.imath.kiev.ua/~skolyada/Gorodnik_O...
5,Open Problems in Algebraic Combinatorics - Sam...,https://www.samuelfhopkins.com/OPAC/files/blog...
6,Open problems,https://www.math.pku.edu.cn/teachers/renyx/Hom...
7,SOME OF MY FAVORITE OPEN PROBLEMS Haïm Brezis,https://sites.math.rutgers.edu/~brezis/PUBlica...
8,OPEN PROBLEMS IN GEOMETRY OF CURVES AND ...,https://ghomi.math.gatech.edu/Papers/op.pdf
9,Open Problems in Coding Theory,https://gilkalai.wordpress.com/wp-content/uplo...


In [2]:
# === Google Search results GPT validation ===
from openai import OpenAI
from tqdm import tqdm

client = OpenAI()

system_prompt = """
You will be given a link to a paper or document.
I'm looking for papers or documents that list open problems that are unsolved.
Your job is to open the link to validate whether the provided paper meets my expectations.
Return either True or False only.
"""

g_output = []
for _, row in tqdm(g_df.iterrows(), total=len(g_df)):
    try:
        response = client.responses.create(
            model="gpt-5-mini",
            tools=[{"type": "web_search_preview"}],
            input=system_prompt + f"use this link {row.url}"
        )
        g_output.append(response.output_text)
    except Exception as e:
        print(f"[gpt] {row.url}: {e}")
        g_output.append("False")

g_df_val = g_df.copy()
g_df_val["is_open_problem_list"] = g_output
g_df_val = g_df_val[g_df_val["is_open_problem_list"].astype(str).str.contains("True")]
g_df_val = g_df_val.drop(columns=["is_open_problem_list"])

g_df_val.to_csv("google-op-papers-validated.csv", index=False)
print(f"validated: {len(g_df_val)} / {len(g_df)}")
g_df_val.head()

100%|██████████| 458/458 [1:49:00<00:00, 14.28s/it]  


validated: 203 / 458


,title,url
0,"100 OPEN PROBLEMS Contents 1. Sum-free sets, p...",https://people.maths.ox.ac.uk/greenbj/papers/o...
1,A List of Open Problems Compiled by,https://aimath.org/WWN/polyaschurlax/polyaschu...
2,Open Problems and Projects Stephen Wolfram,https://www.wolframscience.com/openproblems/NK...
3,Some of my favorite open problems,https://www.stat.berkeley.edu/~aldous/Talks/OP...
4,open problems in dynamics and related fields,https://www.imath.kiev.ua/~skolyada/Gorodnik_O...


In [41]:
# === bioRxiv open-problem harvester (Europe PMC API) ===
# Europe PMC indexes biorxiv preprints and provides a clean REST API —
# same pattern as the arxiv cell above, no browser needed.
import requests, time
from tqdm import tqdm
import pandas as pd

EPMC = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"

brx_phrases = [
    "open problems", "open problem",
    "open questions", "open question",
    "unsolved problems", "unsolved problem",
    "open conjectures",
    "problems and conjectures",
    "list of problems", "list of open problems",
    "problem list",
    "problem collection",
    "research problems",
]

def epmc_fetch(phrase, page_size=100, max_pages=5, delay=1.0):
    rows, cursor = [], "*"
    for _ in range(max_pages):
        params = {
            "query": f'TITLE:"{phrase}" AND SRC:PPR',  # PPR = preprints (biorxiv/medrxiv)
            "resultType": "core",
            "pageSize": page_size,
            "cursorMark": cursor,
            "format": "json",
        }
        try:
            r = requests.get(EPMC, params=params, timeout=30,
                             headers={"User-Agent": "open-problem-harvester/1.0 (kevinjk02@snu.ac.kr)"})
            r.raise_for_status()
        except Exception as e:
            print(f"  [{phrase}] error: {e}")
            break

        data = r.json()
        results = data.get("resultList", {}).get("result", [])
        for item in results:
            doi = item.get("doi", "")
            title = item.get("title", "").strip()
            if not doi or not title:
                continue
            # biorxiv DOIs start with 10.1101/
            if doi.startswith("10.1101/"):
                link = f"https://www.biorxiv.org/content/{doi}"
                rows.append({"title": title, "links": link})

        next_cursor = data.get("nextCursorMark", cursor)
        if next_cursor == cursor or len(results) < page_size:
            break
        cursor = next_cursor
        time.sleep(delay)
    return rows

brx_rows = []
for phrase in tqdm(brx_phrases):
    brx_rows.extend(epmc_fetch(phrase))

# dedupe by link
seen, brx_links, brx_titles = set(), [], []
for row in brx_rows:
    if row["links"] in seen:
        continue
    seen.add(row["links"])
    brx_links.append(row["links"])
    brx_titles.append(row["title"])

brx_df = pd.DataFrame({"title": brx_titles, "links": brx_links})
print(f"bioRxiv unique papers: {len(brx_df)}")
brx_df.head(10)


100%|██████████| 13/13 [00:16<00:00,  1.26s/it]

bioRxiv unique papers: 8


,title,links
0,Universal differential equations for systems b...,https://www.biorxiv.org/content/10.1101/2024.1...
1,The combinatorics of discrete time-trees: theo...,https://www.biorxiv.org/content/10.1101/063362
2,Three Open Questions in Polygenic Score Portab...,https://www.biorxiv.org/content/10.1101/2024.0...
3,Retractions and rewards in science: An open qu...,https://www.biorxiv.org/content/10.1101/2022.0...
4,Predicting pose distribution of protein domain...,https://www.biorxiv.org/content/10.1101/2025.0...
5,Senescence: Still an Unsolved Problem of Biology,https://www.biorxiv.org/content/10.1101/739730
6,The unsolved problem of otitis media in indige...,https://www.biorxiv.org/content/10.1101/355982
7,How to measure obesity in public health resear...,https://www.biorxiv.org/content/10.1101/2025.0...


In [45]:
# === bioRxiv GPT validation ===
from openai import OpenAI
from tqdm import tqdm

client = OpenAI()

system_prompt = """
You will be given a link of a paper.
Im looking for papers that list open problems that are unsolved. 
Your job is to open the link to validate whether the provided paper meets my expectations.
Return either True or False only.
"""

brx_output = []
for _, row in tqdm(brx_df.iterrows(), total=len(brx_df)):
    response = client.responses.create(
        model="gpt-5-mini",
        tools=[{"type": "web_search"}],
        input=system_prompt + f"use this link {row.links}"
    )
    brx_output.append(response.output_text)


100%|██████████| 8/8 [03:42<00:00, 27.87s/it]


In [47]:
# === bioRxiv save validated CSV ===
brx_df_val = brx_df.copy()
brx_df_val['is_open_problem_list'] = brx_output

brx_df_val = brx_df_val[brx_df_val['is_open_problem_list'].astype(str).str.contains('True')]
brx_df_val = brx_df_val.drop(columns=['is_open_problem_list'])

brx_df_val.to_csv('biorxiv-op-papers-validated.csv', index=False)
print(f"validated: {len(brx_df_val)} / {len(brx_df)}")
brx_df_val.head()


validated: 8 / 8


,title,links
0,Universal differential equations for systems b...,https://www.biorxiv.org/content/10.1101/2024.1...
1,The combinatorics of discrete time-trees: theo...,https://www.biorxiv.org/content/10.1101/063362
2,Three Open Questions in Polygenic Score Portab...,https://www.biorxiv.org/content/10.1101/2024.0...
3,Retractions and rewards in science: An open qu...,https://www.biorxiv.org/content/10.1101/2022.0...
4,Predicting pose distribution of protein domain...,https://www.biorxiv.org/content/10.1101/2025.0...
